### RAG Pipeline - Data Ingestion to Vector DB

In [8]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [10]:
# Read all pdf inside the directory

def process_all_pdfs(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file}...")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "PDF"

            all_docs.extend(documents)
            print(f"Loaded {len(documents)} Pages")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

all_documents = process_all_pdfs("../data/pdf")

Found 3 PDF files in the directory.

Processing ..\data\pdf\docker.pdf...
Loaded 4 Pages

Processing ..\data\pdf\fastapi.pdf...
Loaded 4 Pages

Processing ..\data\pdf\python.pdf...
Loaded 4 Pages

Total documents loaded: 12


In [11]:
all_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}, page_content='Introduction to Docker\n1. What is Docker?\nDocker is an open-source platform that allows developers to package applications, along with all of\ntheir dependencies, into standardized units called containers. A container includes everything an\napplication needs to run - code, runtime, system tools, libraries, and configuration files - so it behaves\nthe same way regardless of where it is deployed.\nDocker was first released in 2013 and quickly became the stand

In [12]:
# Splitting the documents into smaller chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [13]:
chunks = split_documents(all_documents)
chunks

Split 12 documents into 22 chunks.
Example chunk:
Content: Introduction to Docker
1. What is Docker?
Docker is an open-source platform that allows developers to package applications, along with all of
their dependencies, into standardized units called contain...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-18T09:48:57+00:00', 'source': 'docker.pdf', 'file_path': '..\\data\\pdf\\docker.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'Introduction to Docker', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-18T09:48:57+00:00', 'trapped': '', 'modDate': "D:20260918094857+00'00'", 'creationDate': "D:20260918094857+00'00'", 'page': 0, 'file_type': 'PDF'}, page_content='Introduction to Docker\n1. What is Docker?\nDocker is an open-source platform that allows developers to package applications, along with all of\ntheir dependencies, into standardized units called containers. A container includes everything an\napplication needs to run - code, runtime, system tools, libraries, and configuration files - so it behaves\nthe same way regardless of where it is deployed.\nDocker was first released in 2013 and quickly became the stand